# Run 2 — YOLO11m Optimized

**Epic:** TTV-118 | **Config:** `yolo11m_optimized.yaml`

Cambios vs Run 1: mixup=0.1, cutmix=0.1, cls_pw=0.5

**Hipótesis:** mixup + cutmix + cls_pw reducido mejoran clases minoritarias

---

## 0. Verificar GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "ERROR: No GPU detectada. Ve a Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Instalar dependencias

In [ ]:
!pip install -q roboflow ultralytics

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/cycling-photo-ai/experiments'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Output dir: {DRIVE_OUTPUT}")

## 3. Descargar dataset v1

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "xOdnFACkI2vaUzBKVRic"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("titan-ca4ce").project("titan-detection-jedpa")
version = project.version(7)

dataset = version.download("yolov11", location="/content/dataset_v1")
print("Dataset descargado")

## 4. Verificar dataset

In [ ]:
from pathlib import Path

dataset_dir = Path("/content/dataset_v1")
for split in ["train", "valid", "test"]:
    imgs = list((dataset_dir / split / "images").glob("*.*"))
    lbls = list((dataset_dir / split / "labels").glob("*.txt"))
    print(f"{split}: {len(imgs)} images, {len(lbls)} labels")

## 5. Reproducibilidad

In [ ]:
import os, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print(f"Seed: {SEED}")

## 6. Entrenar YOLO11m — Run 2 Optimized

Cambios vs Run 1:
- `mixup`: 0.0 → **0.1**
- `cutmix`: 0.0 → **0.1**
- `cls_pw`: 1.0 → **0.5** (reduce penalización clases mayoritarias)

In [ ]:
from ultralytics import YOLO

RUN_NAME = "run2_yolo11m_optimized"

model = YOLO("yolo11m.pt")

results = model.train(
    data="/content/dataset_v1/data.yaml",
    epochs=200,
    patience=30,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=7.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.0,
    flipud=0.0,
    mosaic=1.0,
    close_mosaic=10,
    mixup=0.1,     # CHANGED: 0.0 → 0.1
    cutmix=0.1,    # CHANGED: 0.0 → 0.1
    cls_pw=0.5,    # CHANGED: 1.0 → 0.5
    save_json=True,
    deterministic=True,
    seed=SEED,
    project="/content/experiments",
    name=RUN_NAME,
)

## 7. Revisar resultados

In [ ]:
import pandas as pd

run_dir = Path(f"/content/experiments/{RUN_NAME}")

print("=== Métricas finales ===")
for key, val in results.results_dict.items():
    print(f"  {key}: {val:.4f}")

results_csv = run_dir / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f"\nEpochs entrenados: {len(df)}")
    print(f"Mejor mAP@0.5: {df['metrics/mAP50(B)'].max():.4f} (epoch {df['metrics/mAP50(B)'].idxmax()})")
    print(f"Mejor mAP@0.5:0.95: {df['metrics/mAP50-95(B)'].max():.4f} (epoch {df['metrics/mAP50-95(B)'].idxmax()})")

In [ ]:
import matplotlib.pyplot as plt

if results_csv.exists():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(df['train/box_loss'], label='train box')
    axes[0].plot(df['train/cls_loss'], label='train cls')
    axes[0].plot(df['val/box_loss'], label='val box', linestyle='--')
    axes[0].plot(df['val/cls_loss'], label='val cls', linestyle='--')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[0].set_xlabel('Epoch')

    axes[1].plot(df['metrics/mAP50(B)'], label='mAP@0.5')
    axes[1].plot(df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
    axes[1].axhline(y=0.80, color='r', linestyle=':', label='Target 0.80')
    axes[1].set_title('mAP')
    axes[1].legend()
    axes[1].set_xlabel('Epoch')

    axes[2].plot(df['metrics/precision(B)'], label='Precision')
    axes[2].plot(df['metrics/recall(B)'], label='Recall')
    axes[2].set_title('Precision / Recall')
    axes[2].legend()
    axes[2].set_xlabel('Epoch')

    plt.tight_layout()
    plt.savefig(run_dir / 'training_curves.png', dpi=150)
    plt.show()

## 8. Validación per-class

In [ ]:
best_model = YOLO(str(run_dir / "weights" / "best.pt"))
val_results = best_model.val(data="/content/dataset_v1/data.yaml", imgsz=640, save_json=True)

class_names = ['bicycle', 'bicycle_text', 'clothes_text', 'competidor_number', 'cyclist',
               'cyclist_clothes', 'cyclist_with_bike', 'helmet', 'helmet_text', 'objects']

print("\n=== Per-class AP@0.5 ===")
for i, name in enumerate(class_names):
    ap50 = val_results.box.ap50[i] if i < len(val_results.box.ap50) else 0
    ap = val_results.box.ap[i] if i < len(val_results.box.ap) else 0
    print(f"  {name:25s} AP@0.5={ap50:.4f}  AP@0.5:0.95={ap:.4f}")

## 9. Confusion matrix + PR curves

In [ ]:
from IPython.display import Image, display

for fname, title in [("confusion_matrix_normalized.png", "Confusion Matrix"),
                      ("PR_curve.png", "PR Curves"),
                      ("val_batch0_pred.jpg", "Sample predictions")]:
    fpath = run_dir / fname
    if fpath.exists():
        print(f"\n{title}:")
        display(Image(filename=str(fpath), width=800))

## 10. Guardar en Google Drive

In [ ]:
import shutil

drive_run_dir = Path(DRIVE_OUTPUT) / RUN_NAME
if drive_run_dir.exists():
    shutil.rmtree(drive_run_dir)
shutil.copytree(run_dir, drive_run_dir)

weights_size = (drive_run_dir / "weights" / "best.pt").stat().st_size / 1e6
print(f"Guardado en: {drive_run_dir}")
print(f"best.pt: {weights_size:.1f} MB")

## 11. Resumen para EXPERIMENT_LOG.md

In [ ]:
print("="*60)
print("RESUMEN PARA EXPERIMENT_LOG.md")
print("="*60)
print(f"\n### Run 2 — YOLO11m Optimized")
print(f"- **Fecha:** {pd.Timestamp.now().strftime('%Y-%m-%d')}")
print(f"- **Config:** yolo11m_optimized.yaml")
print(f"- **Dataset:** v1 (sin flip), Roboflow v7")
print(f"- **GPU:** {torch.cuda.get_device_name(0)}")
print(f"- **Cambios vs Run 1:** mixup 0→0.1, cutmix 0→0.1, cls_pw 1.0→0.5")
print(f"- **Epochs entrenados:** {len(df)}")
print(f"- **Mejor epoch:** {df['metrics/mAP50(B)'].idxmax()}")
print(f"")
print(f"| Métrica | Valor |")
print(f"|---|---|")
print(f"| mAP@0.5 | {df['metrics/mAP50(B)'].max():.4f} |")
print(f"| mAP@0.5:0.95 | {df['metrics/mAP50-95(B)'].max():.4f} |")
print(f"| Precision | {df['metrics/precision(B)'].max():.4f} |")
print(f"| Recall | {df['metrics/recall(B)'].max():.4f} |")
print(f"")
print(f"**Per-class AP@0.5:**")
print(f"")
print(f"| Clase | AP@0.5 |")
print(f"|---|---|")
for i, name in enumerate(class_names):
    ap50 = val_results.box.ap50[i] if i < len(val_results.box.ap50) else 0
    print(f"| {name} | {ap50:.4f} |")
print(f"\nPesos guardados en: {drive_run_dir}/weights/best.pt")